# AIE1903 HW1 · 单向 LSTM 情感分类与积分梯度分析

按顺序运行。将 Notebook 与 `lstm.py` 放在项目根目录，将 `train.tsv`、`dev.tsv`、`test.tsv` 放入 `data/SST2/`。

初始化材料满足一项即可：① `data/embeddings_initial.npz` 与 `data/vocab.json`；② 已有 GoogleNews `.bin` 或 `.bin.gz`（放在 `data/` 或项目根目录，读取时需要 `train_batch.py`）。程序优先使用完整矩阵缓存，不自动下载 GoogleNews。

训练、验证选模、测试、正面类 IG 与删词逻辑沿用原实现。实验结果写入 `/tmp/aie1903/runs`；会话结束前下载结果。

## 1. 环境、路径与随机种子配置

输入统一使用项目 `data/`：先检查三个 SST-2 TSV，再检查训练前初始化缓存或已有 GoogleNews 文件。缓存完整时优先复用缓存；否则优先使用未压缩 `.bin`，最后使用 `.bin.gz`。本节不下载任何外部材料。结果继续写入 `/tmp/aie1903/runs`。

In [ ]:
from pathlib import Path
import ast
import random
import importlib
import zipfile
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from types import SimpleNamespace

# 输入文件统一放入项目 data/；输出仍写入可写临时目录。
root = Path.cwd()
data_dir = root / 'data'
sst2_dir = data_dir / 'SST2'
work_root = Path('/tmp/aie1903')
work_root.mkdir(parents=True, exist_ok=True)

assert (root / 'lstm.py').is_file(), f'缺少模型脚本：{root / "lstm.py"}'
missing = [sst2_dir / f'{split}.tsv' for split in ['train', 'dev', 'test']
           if not (sst2_dir / f'{split}.tsv').is_file()]
if missing:
    raise FileNotFoundError('缺少 SST-2 数据，请解压后放入 data/SST2/：\n' + '\n'.join(map(str, missing)))

# 先检查已有初始化材料；完整缓存优先，避免读取巨大向量文件。
initial_matrix_path = data_dir / 'embeddings_initial.npz'
initial_vocab_path = data_dir / 'vocab.json'
vector_candidates = [
    data_dir / 'GoogleNews-vectors-negative300.bin',
    root / 'GoogleNews-vectors-negative300.bin',
]
archive_candidates = [
    data_dir / 'GoogleNews-vectors-negative300.bin.gz',
    root / 'GoogleNews-vectors-negative300.bin.gz',
]
vector_file = next((p for p in vector_candidates if p.is_file()), None)
archive_file = next((p for p in archive_candidates if p.is_file()), None)
if initial_matrix_path.is_file() and initial_vocab_path.is_file():
    initialization_source = 'cache'
    print('初始化材料：使用 data/ 中已有矩阵与词表；无需 GoogleNews 原文件')
elif vector_file is not None or archive_file is not None:
    initialization_source = 'googlenews'
    if not (root / 'train_batch.py').is_file():
        raise FileNotFoundError('读取 GoogleNews 需要项目中的 train_batch.py')
    print('初始化材料：使用已有 GoogleNews 文件：', vector_file or archive_file)
    if initial_matrix_path.is_file() or initial_vocab_path.is_file():
        print('提示：初始化缓存缺少配套文件，本次改用 GoogleNews。')
else:
    raise FileNotFoundError(
        '缺少初始化材料。请提供 data/embeddings_initial.npz 与 data/vocab.json；'
        '或将 GoogleNews-vectors-negative300.bin（或 .bin.gz）放入 data/ 或项目根目录。'
        '程序不会自动下载。'
    )

# 固定全局随机种子以减少随机差异
seed = 1
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# 检查 GPU 可用性与硬件配置
assert torch.cuda.is_available(), '当前 Jupyter 内核没有可用 GPU；请切换至 GPU 节点'
device = torch.device('cuda')
print('项目目录：', root)
print('输出目录：', work_root)
print('PyTorch：', torch.__version__, 'GPU：', torch.cuda.get_device_name(0))

## 2. 数据加载与词表构建

**代码说明**：本节读取 SST-2 文本数据集划分。采用空格切词，保留原始大小写与标点。仅基于训练集构建词表索引映射（包含 `<pad>` 与 `<unk>` 特殊标记）。

In [ ]:
datasets = {}

# 逐个读取 train、dev、test 数据集划分
for split in ["train", "dev", "test"]:
    samples = []
    with open(sst2_dir / f"{split}.tsv", encoding="utf-8") as f:
        for line in f:
            text, label = line.rstrip("\n").rsplit("\t", 1)
            samples.append((text.split(), int(label)))
    datasets[split] = samples

# 验证数据集划分样本格式与非空性
assert all(label in (0, 1) and words for split in datasets.values() for words, label in split)
print('数据量统计：', {k: len(v) for k, v in datasets.items()})

# 初始化词表，定义填充符与未知词标记索引
vocab = {"<pad>": 0, "<unk>": 1}

# 仅使用训练集文本构建词表索引映射
for words, label in datasets["train"]:
    for word in words:
        if word not in vocab:
            vocab[word] = len(vocab)

print('词表总规模：', len(vocab))

## 3. 动态批次构建与右侧填充

**代码说明**：定义 `make_batch` 批次处理函数，将文本序列转换为张量并执行动态右填充（Padding）。

In [ ]:
def make_batch(samples):
    # 将单词序列映射为词表索引张量
    sequences = [torch.tensor([vocab.get(w, vocab['<unk>']) for w in words], dtype=torch.long)
                 for words, _ in samples]
    # 动态右侧填充，形状为 [最大句长, 批量大小]
    text = pad_sequence(sequences, batch_first=False, padding_value=vocab['<pad>']).to(device)
    # 标签映射调整以适配接口
    label = torch.tensor([label + 1 for _, label in samples], dtype=torch.long, device=device)
    return SimpleNamespace(text=text, label=label)

# 构建训练集、验证集与测试集的 DataLoader 迭代器
train_iter = DataLoader(datasets['train'], batch_size=5, shuffle=True, collate_fn=make_batch)
dev_iter = DataLoader(datasets['dev'], batch_size=64, shuffle=False, collate_fn=make_batch)
test_iter = DataLoader(datasets['test'], batch_size=64, shuffle=False, collate_fn=make_batch)

# 检查样例批次形状是否正确
batch = next(iter(train_iter))
assert batch.text.shape[1] == 5 and batch.label.shape == (5,)
print('样例批次形状：', tuple(batch.text.shape), tuple(batch.label.shape))

## 4. LSTM 模型架构与填充感知扩展

**代码说明**：继承自原始 `lstm.py` 中的网络结构，在 Notebook 内部覆盖 `forward` 方法，引入 `pack_padded_sequence` 机制来动态跳过右侧填充符号，确保长短期记忆网络计算的准确性。

In [ ]:
import lstm
importlib.reload(lstm)

class PackedLSTMSentiment(lstm.LSTMSentiment):
    def forward(self, sentence):
        # 词嵌入映射
        x = self.embeddings(sentence)
        # 计算每个样本的真实非填充句长
        lengths = sentence.ne(vocab['<pad>']).sum(dim=0)
        # 打包填充序列以高效送入 LSTM
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), enforce_sorted=False)
        _, self.hidden = self.lstm(packed, self.hidden)
        # 映射至二分类对数概率输出
        y = self.hidden2label(self.hidden[0][-1])
        return F.log_softmax(y, dim=1)

# 实例化模型（输入维度 300，隐藏层 150，类别数 2）
model = PackedLSTMSentiment(300, 150, len(vocab), 2, True, 5).to(device)
model.batch_size = len(batch.label)
model.hidden = model.init_hidden()

with torch.no_grad():
    assert model(batch.text).shape == (5, 2)
print('LSTM 前向传播形状检查通过')

## 5. 训练前嵌入矩阵初始化

优先加载 `data/embeddings_initial.npz` 和 `data/vocab.json`，核对词表与矩阵格式。没有完整缓存时，读取第 1 节找到的 GoogleNews `.bin`；仅有 `.bin.gz` 时解压到临时目录。同一来源的已完成解压缓存可复用，解压失败不会保留为完整文件。初始化材料不来自训练后的模型参数。

In [ ]:
import json

initial_matrix_path = data_dir / 'embeddings_initial.npz'
initial_vocab_path = data_dir / 'vocab.json'

# 检查本地是否存在预先构建的嵌入初始化缓存
if initialization_source == 'cache':
    saved_vocab = json.loads(initial_vocab_path.read_text(encoding='utf-8'))
    assert saved_vocab == vocab, '初始化矩阵的词表与训练集词表不一致'
    with np.load(initial_matrix_path, allow_pickle=False) as cache:
        weights = cache['weights'].copy()
        vector_match_count = int(cache['matched_count'].item())
    assert weights.shape == (len(vocab), 300)
    assert weights.dtype == np.float32 and np.isfinite(weights).all()
    assert np.all(weights[vocab['<pad>']] == 0)
    with torch.no_grad():
        model.embeddings.weight.copy_(torch.from_numpy(weights).to(device))
    print('已成功加载训练前初始化矩阵缓存')
else:
    import gzip
    import shutil

    # 第 1 节已确认材料路径。已有 .bin 时直接读取，不重复解压。
    if vector_file is None:
        import hashlib
        source_stat = archive_file.stat()
        cache_key = hashlib.sha256(
            f'{archive_file.resolve()}:{source_stat.st_size}:{source_stat.st_mtime_ns}'.encode('utf-8')
        ).hexdigest()[:16]
        vector_file = work_root / 'vector_cache' / cache_key / 'GoogleNews-vectors-negative300.bin'
        if not vector_file.is_file():
            vector_file.parent.mkdir(parents=True, exist_ok=True)
            partial_file = vector_file.with_suffix('.bin.partial')
            try:
                with gzip.open(archive_file, 'rb') as source, partial_file.open('wb') as target:
                    shutil.copyfileobj(source, target)
                partial_file.replace(vector_file)
            except Exception:
                if partial_file.exists():
                    partial_file.unlink()
                raise
        print('GoogleNews 解压缓存：', vector_file)
    if vector_file.stat().st_size == 0:
        raise ValueError(f'GoogleNews 文件为空：{vector_file}')

    tree = ast.parse((root / 'train_batch.py').read_text(encoding='utf-8'))
    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == 'load_bin_vec')
    namespace = {'np': np, 'numpy': np}
    exec(compile(ast.Module(body=[node], type_ignores=[]), 'train_batch.py', 'exec'), namespace)
    found = namespace['load_bin_vec'](str(vector_file), vocab)

    weights = np.random.uniform(-0.25, 0.25, size=(len(vocab), 300)).astype('float32')
    weights[vocab['<pad>']] = 0
    for word, vector in found.items():
        weights[vocab[word]] = vector
    with torch.no_grad():
        model.embeddings.weight.copy_(torch.from_numpy(weights).to(device))
    vector_match_count = len(found)
    print(f'匹配tokens数：{len(found)} / {len(vocab) - 2}；嵌入矩阵：{tuple(model.embeddings.weight.shape)}')

## 6. 模型训练与验证过程

**代码说明**：设定训练轮次（Epochs），采用 Adam 优化器与损失函数进行训练。每个 Epoch 结束后在验证集上评估，记录损失与准确率，并基于验证集表现动态保存最优模型检查点权重。

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.NLLLoss()
run_dir = work_root / 'runs'
run_dir.mkdir(exist_ok=True)
checkpoint_path = run_dir / 'final_best_lstm.pt'

def evaluate(loader):
    model.eval()
    correct = count = 0
    with torch.no_grad():
        for current in loader:
            labels = current.label - 1
            model.batch_size = len(labels)
            model.hidden = model.init_hidden()
            prediction = model(current.text)
            correct += (prediction.argmax(dim=1) == labels).sum().item()
            count += len(labels)
    return correct, count

history = []
best_dev_accuracy, best_epoch = -1, None

# 循环进行训练与验证
for epoch in range(1, 6):
    model.train()
    total_loss = total_correct = total_count = 0
    for current in train_iter:
        labels = current.label - 1
        model.batch_size = len(labels)
        model.hidden = model.init_hidden()
        optimizer.zero_grad()
        prediction = model(current.text)
        loss = loss_fn(prediction, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total_correct += (prediction.argmax(dim=1) == labels).sum().item()
        total_count += len(labels)
    
    dev_correct, dev_count = evaluate(dev_iter)
    dev_accuracy = dev_correct / dev_count
    history.append((epoch, total_loss / total_count, total_correct / total_count, dev_accuracy))
    
    # 保存验证集表现最优的模型权重参数
    if dev_accuracy > best_dev_accuracy:
        best_dev_accuracy, best_epoch = dev_accuracy, epoch
        torch.save({'model_state': model.state_dict(), 'vocab': vocab,
                    'epoch': epoch, 'dev_accuracy': dev_accuracy}, checkpoint_path)
    print(f'第{epoch}轮：loss={history[-1][1]:.4f}，训练准确率={history[-1][2]:.1%}，'
          f'验证准确率={dev_accuracy:.1%}；当前最佳：第{best_epoch}轮 {best_dev_accuracy:.1%}', flush=True)

## 7. 测试集评估与解释案例选择

**代码说明**：加载验证集选出的最佳模型检查点权重，在独立测试集上进行最终评估，并按照固定规则从测试集中筛选出代表性解释案例。

In [ ]:
# 加载本次训练选出的最佳模型检查点
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True,
)
assert checkpoint["vocab"] == vocab
model.load_state_dict(checkpoint["model_state"])
model.eval()

# 独立测试集评估
correct, count = evaluate(test_iter)
print(
    f"最佳轮次：{checkpoint['epoch']}；"
    f"测试集准确率：{correct / count:.1%} ({correct}/{count})"
)

# 遍历测试集记录每条影评的真实标签、预测类别与置信度
records = []
with torch.no_grad():
    for current in test_iter:
        model.batch_size = len(current.label)
        model.hidden = model.init_hidden()

        probabilities = model(current.text).exp()
        confidence, predicted = probabilities.max(dim=1)

        for j in range(len(current.label)):
            index = len(records)
            words, true_label = datasets["test"][index]
            records.append({
                "index": index,
                "words": words,
                "true": true_label,
                "pred": predicted[j].item(),
                "confidence": confidence[j].item(),
            })

assert len(records) == count
assert sum(r["true"] == r["pred"] for r in records) == correct

# 计算正面类的 Precision、Recall 与 F1 值指标
tp = sum(r["true"] == 1 and r["pred"] == 1 for r in records)
fp = sum(r["true"] == 0 and r["pred"] == 1 for r in records)
fn = sum(r["true"] == 1 and r["pred"] == 0 for r in records)

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)
print(
    f"正面类 Precision={precision:.1%}，"
    f"Recall={recall:.1%}，F1={f1:.1%}"
)

# 定义五类候选池及固定的案例挑选规则
categories = [
    (
        "置信度最接近 50%",
        lambda r: True,
        lambda r: (abs(r["confidence"] - 0.50), r["index"]),
    ),
    (
        "预测正确、低置信度（50%–60%）",
        lambda r: (
            r["pred"] == r["true"]
            and 0.50 <= r["confidence"] <= 0.60
        ),
        lambda r: (r["confidence"], r["index"]),
    ),
    (
        "预测正确、中等置信度（65%–80%）",
        lambda r: (
            r["pred"] == r["true"]
            and 0.65 <= r["confidence"] <= 0.80
        ),
        lambda r: (abs(r["confidence"] - 0.725), r["index"]),
    ),
    (
        "预测错误、高置信度（≥90%）",
        lambda r: (
            r["pred"] != r["true"]
            and r["confidence"] >= 0.90
        ),
        lambda r: (-r["confidence"], r["index"]),
    ),
    (
        "预测错误、中等置信度（65%–80%）",
        lambda r: (
            r["pred"] != r["true"]
            and 0.65 <= r["confidence"] <= 0.80
        ),
        lambda r: (abs(r["confidence"] - 0.725), r["index"]),
    ),
]

pools = [
    (name, [r for r in records if condition(r)], ranking)
    for name, condition, ranking in categories
]

# 按照固定顺序与规则提取 5 个不重复的代表性样本
selected = []
selection_reasons = []
used = set()

for name, pool, ranking in pools:
    available = [r for r in pool if r["index"] not in used]
    if not available:
        raise ValueError(f"没有可供选择的案例：{name}")

    chosen = min(available, key=ranking)
    selected.append(chosen)
    selection_reasons.append(name)
    used.add(chosen["index"])

case_ids = [r["index"] for r in selected]
assert len(case_ids) == 5 and len(set(case_ids)) == 5

for r, reason in zip(selected, selection_reasons):
    print(
        f"#{r['index']} [{reason}] "
        f"真={r['true']} 预测={r['pred']} "
        f"置信度={r['confidence']:.1%}："
        f"{' '.join(r['words'])}"
    )

## 8. 正面类 Integrated Gradients（积分梯度）可解释性分析

**代码说明**：基于 Captum 库对选定测试样本计算积分梯度。目标固定为正面类对数概率 log p1，基线为全零嵌入，积分步数为 64。正 IG 支持正面输出，负 IG 反对正面输出；`top3` 按 tokens 的 IG 绝对值排序，不代表删词效应排名。

**输出含义解释**：

- **`输出差`（Output Difference）**：真实句子与零嵌入基线分别输入模型后，正面类对数概率相差多少，即 $\text{input\_log\_p1}-\text{baseline\_log\_p1}$。
- **`delta`（残差）**：所有 token 的 IG 值相加后，与输出差相差多少，即 $\delta=\text{sum\_ig}-\text{输出差}$。理论上，归因总和应等于输出差；数值积分可能产生偏差，因此用 `delta` 检查总和是否对齐。
- **`相对残差`（Relative Residual）**：残差绝对值占输出差绝对值多大比例，即 $\lvert\delta\rvert/\lvert\text{输出差}\rvert$。**较小相对残差说明归因总和对齐较好。但不能保证每个 token 的归因都准确。**当输出差接近零时，相对残差可能很大，需要结合 $\lvert\delta\rvert$ 判断。


In [ ]:
from captum.attr import IntegratedGradients

IG_TARGET = 1
IG_STEPS = 64
DIRECTION_EPS = 1e-6

assert len(case_ids) == 5 and len(set(case_ids)) == 5
assert checkpoint['vocab'] == vocab
model.load_state_dict(checkpoint['model_state'])
model.eval()

def score_from_embeddings(emb):
    x = emb.transpose(0, 1)
    batch_size = emb.shape[0]
    h0 = x.new_zeros(1, batch_size, model.hidden_dim)
    c0 = x.new_zeros(1, batch_size, model.hidden_dim)
    _, (h, _) = model.lstm(x, (h0, c0))
    return F.log_softmax(model.hidden2label(h[-1]), dim=1)

ig = IntegratedGradients(score_from_embeddings)
ig_results = {}

# 针对精选案例执行积分梯度计算
for index in case_ids:
    r = records[index]
    ids = torch.tensor(
        [vocab.get(w, vocab['<unk>']) for w in r['words']],
        dtype=torch.long, device=device,
    ).unsqueeze(1)
    emb = model.embeddings(ids).transpose(0, 1).detach()
    baseline = torch.zeros_like(emb)
    
    with torch.no_grad(), torch.backends.cudnn.flags(enabled=False):
        model.batch_size = 1
        model.hidden = model.init_hidden()
        direct = model(ids)
        via_embedding = score_from_embeddings(emb)
        assert torch.allclose(direct, via_embedding, atol=1e-5), (
            f'#{index} IG 入口与模型输出不一致'
        )
        assert direct.argmax(dim=1).item() == r['pred'], (
            '模型或第 7 节结果已改变，请先重新运行第 7 节'
        )
        input_output = via_embedding[0, IG_TARGET].item()
        baseline_output = score_from_embeddings(baseline)[0, IG_TARGET].item()
    
    with torch.backends.cudnn.flags(enabled=False):
        attr, delta = ig.attribute(
            emb, baselines=baseline, target=IG_TARGET,
            n_steps=IG_STEPS, internal_batch_size=16,
            return_convergence_delta=True,
        )
    
    scores = attr.sum(dim=2).squeeze(0).detach().cpu().tolist()
    assert len(scores) == len(r['words'])
    top3 = sorted(range(len(scores)), key=lambda i: (-abs(scores[i]), i))[:3]
    output_difference = input_output - baseline_output
    
    ig_results[index] = {
        'scores': scores, 'target': IG_TARGET, 'n_steps': IG_STEPS,
        'delta': delta.item(), 'top3_positions': top3,
        'input_log_p1': input_output, 'baseline_log_p1': baseline_output,
        'output_difference': output_difference,
        'sum_ig': float(sum(scores)),
        'relative_residual': abs(delta.item()) / max(abs(output_difference), 1e-8),
    }
    result = ig_results[index]
    print(
        f"\n#{index}：目标=正面类；delta={result['delta']:+.8f}；"
        f"输出差={output_difference:+.6f}；"
        f"相对残差={result['relative_residual']:.3%}"
    )
    for rank, pos in enumerate(top3, 1):
        print(f"{rank}. 位置={pos+1}；tokens={r['words'][pos]!r}；IG={scores[pos]:+.6f}")

## 9. 删词检验（Deletion Experiment）

**代码说明**：每句删除纯字母 tokens 中 IG 绝对值最高的一个位置，并比较删除前后模型输出。重复 tokens 按位置分别处理。删词可能改变句意、句长及 LSTM 状态，因此删词结果作为归因解释的补充检验。

**输出含义解释**：

- **`p1`（正面类概率）**：$p_1=P(y=1\mid x)$，格式为“删词前概率 → 删词后概率”。
- **`预测`（Prediction）**：0 代表负面，1 代表正面，格式为“删词前类别 → 删词后类别”。
- **`置信度`（Confidence）**：$\max(p_1,1-p_1)$，格式为“删词前置信度 → 删词后置信度”。
- **`IG`**：被删除 token 对正面类对数概率的归因。
- **`Δp1`（百分点）**：$100[p_1(\text{删词后})-p_1(\text{原句})]$。
- **`d1`（正面类对数概率下降量）**：$\log p_1(\text{原句})-\log p_1(\text{删词后})$。方向判断比较 IG 与 d1 的符号。
- **`一致性方向标记`**：
  - `same_sign`（同号一致）：IG 与 d1 同号，即正 IG 对应删词后 p1 下降，或负 IG 对应删词后 p1 上升。
  - `opposite_sign`（方向相反）：IG 与 d1 异号，即删词变化与上述预期相反。
  - `near_zero`（接近零）：IG 或 d1 的绝对值小于 `DIRECTION_EPS=1e-6`，方向不作明确判断。阈值针对 IG 与 d1，而非概率差。


In [ ]:
def predict_log_words(words):
    assert words, '输入句子不能为空'
    ids = torch.tensor(
        [vocab.get(w, vocab['<unk>']) for w in words],
        dtype=torch.long, device=device,
    ).unsqueeze(1)
    with torch.no_grad(), torch.backends.cudnn.flags(enabled=False):
        emb = model.embeddings(ids).transpose(0, 1)
        return score_from_embeddings(emb).squeeze(0).detach().cpu()

model.eval()
deletion_results = []

# 逐个案例执行删词并评估预测变化
for index, reason in zip(case_ids, selection_reasons):
    r = records[index]
    result = ig_results[index]
    assert result['target'] == 1, '请先运行第 8 节'
    words, scores = r['words'], result['scores']
    candidates = [i for i, word in enumerate(words) if word.isalpha()]
    if not candidates or len(words) < 2:
        raise ValueError(f'#{index} 无法按既定规则删词')
    
    position = max(candidates, key=lambda i: (abs(scores[i]), -i))
    edited_words = words[:position] + words[position + 1:]
    
    before_log = predict_log_words(words)
    after_log = predict_log_words(edited_words)
    before, after = before_log.exp(), after_log.exp()
    
    assert before_log.argmax().item() == r['pred']
    assert abs(before_log[1].item() - result['input_log_p1']) < 1e-4
    
    ig_value = scores[position]
    d1 = (before_log[1] - after_log[1]).item()
    direction = (
        'near_zero' if abs(ig_value) < DIRECTION_EPS or abs(d1) < DIRECTION_EPS
        else 'same_sign' if ig_value * d1 > 0 else 'opposite_sign'
    )
    
    item = {
        'index': index, 'reason': reason, 'true': r['true'], 'target': 1,
        'original_text': ' '.join(words), 'edited_text': ' '.join(edited_words),
        'position': position, 'position_1based': position + 1,
        'word': words[position], 'ig': ig_value,
        'p1_before': before[1].item(), 'p1_after': after[1].item(),
        'log_p1_before': before_log[1].item(), 'log_p1_after': after_log[1].item(),
        'confidence_before': before.max().item(), 'confidence_after': after.max().item(),
        'pred_before': before.argmax().item(), 'pred_after': after.argmax().item(),
        'prob_change_pp': 100 * (after[1] - before[1]).item(),
        'log_drop': d1, 'direction': direction,
        'flipped': before.argmax().item() != after.argmax().item(),
    }
    deletion_results.append(item)
    print(
        f"#{index} 删除 {item['word']!r}（位置 {position+1}）；"
        f"p1 {item['p1_before']:.2%} → {item['p1_after']:.2%}；"
        f"预测 {item['pred_before']} → {item['pred_after']}；"
        f"置信度 {item['confidence_before']:.2%} → {item['confidence_after']:.2%}；"
        f"IG={ig_value:+.8f}；"
        f"Δp1={item['prob_change_pp']:+.8f} pp；"
        f"d1={d1:+.8f}；{direction}"
    )

## 10A. 初始化矩阵与词表本地保存

本节只保存初始化矩阵和词表，不提供下载按钮。优先保存到项目 `data/`；若目录权限不足，则保存到临时目录并提示最终提交时需复制文件。变量 `weights` 必须仍是第 5 节初始化矩阵。

In [ ]:
# 保存初始化矩阵与词表，不打包下载
from pathlib import Path
from io import BytesIO
import json
import base64
import platform
import hashlib
import zipfile
import numpy as np
import torch
from IPython.display import display, HTML

# 可写时直接保存到 data/，便于下次运行；只读挂载则使用原临时目录。
import tempfile
init_dir = data_dir
try:
    init_dir.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryFile(dir=init_dir) as probe:
        probe.write(b'write-check')
except PermissionError:
    init_dir = Path(run_dir) / 'initialization_materials'
    init_dir.mkdir(parents=True, exist_ok=True)
init_files = []

def init_save_json(name, data):
    path = init_dir / name
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    init_files.append(path)

def initialization_fingerprint(matrix, vocabulary):
    matrix = np.ascontiguousarray(matrix, dtype='<f4')
    digest = hashlib.sha256()
    digest.update(json.dumps(vocabulary, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8'))
    digest.update(str(matrix.shape).encode('ascii'))
    digest.update(matrix.tobytes(order='C'))
    return digest.hexdigest()

if 'weights' in globals():
    initial_weights = np.asarray(weights)
    assert initial_weights.shape == (len(vocab), model.embeddings.embedding_dim)
    assert initial_weights.dtype == np.float32 and np.isfinite(initial_weights).all()
    assert np.all(initial_weights[vocab['<pad>']] == 0)
    matched_count = int(vector_match_count) if 'vector_match_count' in globals() else (len(found) if 'found' in globals() else -1)
    initial_path = init_dir / 'embeddings_initial.npz'
    np.savez_compressed(initial_path, weights=initial_weights, matched_count=np.array(matched_count, dtype=np.int64), initialization_sha256=np.array(initialization_fingerprint(initial_weights, vocab)))
    init_files.append(initial_path)
    init_save_json('vocab.json', vocab)
    print('初始化矩阵与词表保存目录：', init_dir)
    if init_dir != data_dir:
        print('项目 data/ 不可写；最终提交前请手动将两个文件复制到提交包的 data/。')
else:
    print('10A：未检测到 weights 变量，跳过临时导出。')

## 10. 本次实验结果与图表打包下载

本节导出训练记录、测试指标与预测、IG 与删词数据、七张图表、字段说明及最优权重。运行配置 `run_config.json` 只保存在服务器，不加入下载 ZIP。输出仅包含成功说明与一个结果下载按钮。


In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
import csv
import json
import base64
import platform
import hashlib
import zipfile
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML

assert len(deletion_results) == len(case_ids) == 5
assert all(ig_results[i]['target'] == 1 for i in case_ids)
assert all(item['target'] == 1 for item in deletion_results)
assert {item['index'] for item in deletion_results} == set(case_ids)
assert len(records) == count
assert sum(r['true'] == r['pred'] for r in records) == correct

# 创建导出目录与图表子目录
export_dir = Path(run_dir) / 'positive_class_results'
fig_dir = export_dir / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
export_files = []

def save_json(name, data, include_in_zip=True):
    path = export_dir / name
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    if include_in_zip:
        export_files.append(path)

def save_csv(name, rows):
    assert rows, f'{name} 没有数据'
    path = export_dir / name
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    export_files.append(path)

def initialization_fingerprint(matrix, vocabulary):
    matrix = np.ascontiguousarray(matrix, dtype='<f4')
    digest = hashlib.sha256()
    digest.update(json.dumps(vocabulary, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8'))
    digest.update(str(matrix.shape).encode('ascii'))
    digest.update(matrix.tobytes(order='C'))
    return digest.hexdigest()

matched_count = int(vector_match_count) if 'vector_match_count' in globals() else (len(found) if 'found' in globals() else None)
if 'weights' in globals():
    initial_sha256 = initialization_fingerprint(weights, vocab)
elif (data_dir / 'embeddings_initial.npz').is_file():
    with np.load(data_dir / 'embeddings_initial.npz', allow_pickle=False) as cache:
        initial_sha256 = initialization_fingerprint(cache['weights'], vocab)
else:
    initial_sha256 = None

# 1. 导出训练历史记录
save_csv('training_history.csv', [
    {'epoch': ep, 'train_loss': loss, 'train_accuracy': train_acc, 'dev_accuracy': dev_acc}
    for ep, loss, train_acc, dev_acc in history
])

# 2. 导出测试集预测结果
prediction_rows = []
for r in records:
    p1 = r['confidence'] if r['pred'] == 1 else 1.0 - r['confidence']
    prediction_rows.append({
        'index': r['index'], 'text': ' '.join(r['words']),
        'true': r['true'], 'pred': r['pred'], 'p1': p1,
        'confidence': r['confidence'], 'correct': r['true'] == r['pred'],
    })
save_csv('test_predictions.csv', prediction_rows)

metrics = {
    'checkpoint_epoch': int(checkpoint['epoch']),
    'dev_accuracy': float(checkpoint['dev_accuracy']),
    'test_correct': correct, 'test_count': count, 'test_accuracy': correct / count,
    'positive_precision': precision, 'positive_recall': recall, 'positive_f1': f1,
    'tp': tp, 'fp': fp, 'fn': fn,
    'tn': sum(r['true'] == 0 and r['pred'] == 0 for r in records),
    'deletion_case_count': 5,
    'direction_same_sign_count': sum(r['direction'] == 'same_sign' for r in deletion_results),
    'direction_opposite_sign_count': sum(r['direction'] == 'opposite_sign' for r in deletion_results),
    'direction_near_zero_count': sum(r['direction'] == 'near_zero' for r in deletion_results),
    'prediction_flip_count': sum(r['flipped'] for r in deletion_results),
}
save_json('test_metrics.json', metrics)

# 3. 导出归因与删词数据
ig_rows, diagnostic_rows = [], []
for index, reason in zip(case_ids, selection_reasons):
    r, result = records[index], ig_results[index]
    ranks = {pos: rank for rank, pos in enumerate(
        sorted(range(len(result['scores'])), key=lambda j: (-abs(result['scores'][j]), j)), 1)}
    for pos, (word, score) in enumerate(zip(r['words'], result['scores'])):
        ig_rows.append({
            'index': index, 'position_1based': pos + 1, 'token': word,
            'vocab_id': vocab.get(word, vocab['<unk>']), 'is_oov': word not in vocab,
            'is_deletion_candidate': word.isalpha(), 'target': 1,
            'ig': score, 'abs_ig_rank': ranks[pos], 'delta': result['delta'],
        })
    diagnostic_rows.append({
        'index': index, 'selection_reason': reason,
        'true': r['true'], 'pred': r['pred'],
        **{k: result[k] for k in [
            'target', 'n_steps', 'delta', 'input_log_p1', 'baseline_log_p1',
            'output_difference', 'sum_ig', 'relative_residual']},
    })
save_csv('ig_attributions.csv', ig_rows)
save_csv('ig_diagnostics.csv', diagnostic_rows)
save_csv('deletion_results.csv', [
    {k: v for k, v in item.items() if k != 'position'} for item in deletion_results
])

# 4. 生成 IG 归因图表
for index in case_ids:
    r = records[index]
    words, scores = r['words'], ig_results[index]['scores']
    limit = max(max(abs(s) for s in scores) * 1.35, 1e-6)
    fig, ax = plt.subplots(figsize=(10, max(5, len(words) * .39 + 1.5)))
    try:
        ax.barh(range(len(words)), scores, color=['#16803c' if s >= 0 else '#7b3294' for s in scores])
        ax.set_yticks(range(len(words)))
        ax.set_yticklabels([f'{j+1:02d}  {w}' for j, w in enumerate(words)])
        ax.invert_yaxis()
        ax.axvline(0, color='black', linewidth=.8)
        ax.set_xlim(-limit, limit)
        ax.set_xlabel('IG attribution to POSITIVE class (log p1)')
        p1 = prediction_rows[index]['p1']
        ax.set_title(f"Test #{index} | true={r['true']} pred={r['pred']} | p1={p1:.2%} confidence={r['confidence']:.2%}")
        for j, s in enumerate(scores):
            ax.text(s + (limit*.015 if s >= 0 else -limit*.015), j, f'{s:+.3f}',
                    va='center', ha='left' if s >= 0 else 'right', fontsize=8)
        fig.tight_layout()
        path = fig_dir / f'ig_{index}.png'
        fig.savefig(path, dpi=200, bbox_inches='tight')
        export_files.append(path)
    finally:
        plt.close(fig)

# 5. 生成删词对比表图
headers = ['Case', 'Token [pos]', 'IG to\npositive', 'p1\nbefore', 'p1\nafter',
           'Confidence\nbefore', 'Confidence\nafter', 'Pred\nbefore -> after',
           'Change\n(pp)', 'd1', 'Direction']
short_direction = {'same_sign': 'Same sign', 'opposite_sign': 'Opposite', 'near_zero': 'Near zero'}
rows = [[f"#{r['index']}", f"{r['word']} [{r['position_1based']}]", f"{r['ig']:+.4f}",
         f"{r['p1_before']:.2%}", f"{r['p1_after']:.2%}",
         f"{r['confidence_before']:.2%}", f"{r['confidence_after']:.2%}",
         f"{r['pred_before']} -> {r['pred_after']}", f"{r['prob_change_pp']:+.2f}",
         f"{r['log_drop']:+.4f}", short_direction[r['direction']]] for r in deletion_results]
fig, ax = plt.subplots(figsize=(20, 4))
try:
    ax.axis('off')
    table = ax.table(cellText=rows, colLabels=headers, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2.3)
    for (row, col), entry in table.get_celld().items():
        if row == 0:
            entry.set_facecolor('#e5edf5')
            entry.set_text_props(weight='bold')
    ax.set_title('Deletion comparison: target always POSITIVE (1)', pad=15)
    fig.text(.5, .04, 'Change = 100*(p1 after - p1 before); d1 = log p1 before - log p1 after. Position is 1-based.', ha='center', fontsize=9)
    fig.tight_layout(rect=(0, .1, 1, 1))
    path = fig_dir / 'deletion_comparison.png'
    fig.savefig(path, dpi=200, bbox_inches='tight')
    export_files.append(path)
finally:
    plt.close(fig)

# 6. 生成方向散点图
fig, ax = plt.subplots(figsize=(8, 6))
try:
    ax.axhline(0, color='black', linewidth=.8)
    ax.axvline(0, color='black', linewidth=.8)
    for direction, color in [('same_sign', '#16803c'), ('opposite_sign', '#7b3294'), ('near_zero', '#777777')]:
        group = [r for r in deletion_results if r['direction'] == direction]
        if group:
            ax.scatter([r['ig'] for r in group], [r['log_drop'] for r in group], s=70, color=color, label=short_direction[direction])
    for j, r in enumerate(deletion_results):
        ax.annotate(f"#{r['index']} {r['word']}", (r['ig'], r['log_drop']),
                    xytext=(6, 8 if j % 2 == 0 else -14), textcoords='offset points', fontsize=9)
    ax.set_xlabel('IG attribution to positive class of deleted token')
    ax.set_ylabel('d1 = log p1(original) - log p1(deleted)')
    ax.set_title('Positive-class IG and deletion: direction comparison')
    ax.legend()
    ax.margins(x=.25, y=.2)
    fig.tight_layout()
    path = fig_dir / 'ig_deletion_direction.png'
    fig.savefig(path, dpi=200, bbox_inches='tight')
    export_files.append(path)
finally:
    plt.close(fig)

# 7. 保存最佳模型权重
best_path = export_dir / 'best_model.pt'
best_cpu = dict(checkpoint)
best_cpu['model_state'] = {k: v.detach().cpu() for k, v in checkpoint['model_state'].items()}
torch.save(best_cpu, best_path)
export_files.append(best_path)
model_sha256 = hashlib.sha256(best_path.read_bytes()).hexdigest()

import captum
save_json('run_config.json', {
    'exported_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(), 'torch': str(torch.__version__),
    'captum': captum.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__,
    'cuda_build': torch.version.cuda, 'device': str(device),
    'gpu': torch.cuda.get_device_name(device) if torch.device(device).type == 'cuda' else None,
    'seed': seed, 'vocab_size': len(vocab),
    'embedding_dim': model.embeddings.embedding_dim, 'hidden_dim': model.hidden_dim,
    'optimizer': 'Adam', 'learning_rate': 1e-3,
    'train_batch_size': train_iter.batch_size, 'epochs': len(history),
    'ig_target': 1, 'ig_output': 'log p(y=1|x)', 'ig_baseline': 'zero embeddings',
    'ig_steps': IG_STEPS, 'ig_method': 'gausslegendre', 'ig_internal_batch_size': 16,
    'direction_epsilon': DIRECTION_EPS, 'selected_case_ids': case_ids,
    'selection_reasons': selection_reasons,
    'deletion_rule': 'max absolute positive-class IG among str.isalpha() tokens; earliest tie',
    'prediction_export': 'p1 reconstructed from binary predicted class and confidence',
    'matched_vector_count': matched_count,
    'initial_embeddings_content_sha256': initial_sha256,
    'initialization_files': ['embeddings_initial.npz', 'vocab.json'],
    'best_model_sha256': model_sha256,
}, include_in_zip=False)
save_json('data_definitions.json', {
    'labels': {'0': 'negative', '1': 'positive'},
    'index': 'zero-based index in original test dataset',
    'position_1based': 'one-based token position; repeated tokens are separate occurrences',
    'p1': 'P(y=1|x)',
    'confidence': 'max(P(y=0|x), P(y=1|x))',
    'ig': 'attribution to log p1 relative to zero embeddings',
    'prob_change_pp': '100 * (p1_after - p1_before)',
    'log_drop': 'log_p1_before - log_p1_after',
    'delta': 'Captum convergence residual',
    'relative_residual': 'abs(delta)/max(abs(output_difference),1e-8)',
})

# 打包 ZIP 并生成下载按钮
zip_buffer = BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in export_files:
        archive.write(path, arcname=path.relative_to(export_dir).as_posix())
zip_bytes = zip_buffer.getvalue()
zip_path = export_dir / 'HW1_experiment_results.zip'
zip_path.write_bytes(zip_bytes)
encoded = base64.b64encode(zip_bytes).decode('ascii')
display(HTML(
    '<p>实验结果创建成功：7 张图表、训练记录、测试与预测、IG、删词数据及最优权重已打包。</p>'
    '<a download="HW1_experiment_results.zip" '
    f'href="data:application/zip;base64,{encoded}" '
    'style="display:inline-block;padding:12px 18px;background:#1769aa;color:white;border-radius:6px;text-decoration:none;">'
    '下载本次实验结果</a>'
))

## 使用与提交说明

请在会话结束前点击上方按钮下载本次运行的实验结果压缩包。